In [1]:
from transformers import AutoModelForCausalLM

In [2]:
gpt2 = AutoModelForCausalLM.from_pretrained('gpt2')

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [3]:
gpt2

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [4]:
type(gpt2)

transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel

In [5]:
gpt2.transformer.wte.weight

Parameter containing:
tensor([[-0.1101, -0.0393,  0.0331,  ..., -0.1364,  0.0151,  0.0453],
        [ 0.0403, -0.0486,  0.0462,  ...,  0.0861,  0.0025,  0.0432],
        [-0.1275,  0.0479,  0.1841,  ...,  0.0899, -0.1297, -0.0879],
        ...,
        [-0.0445, -0.0548,  0.0123,  ...,  0.1044,  0.0978, -0.0695],
        [ 0.1860,  0.0167,  0.0461,  ..., -0.0963,  0.0785, -0.0225],
        [ 0.0514, -0.0277,  0.0499,  ...,  0.0070,  0.1552,  0.1207]],
       requires_grad=True)

In [6]:
gpt2.lm_head.weight

Parameter containing:
tensor([[-0.1101, -0.0393,  0.0331,  ..., -0.1364,  0.0151,  0.0453],
        [ 0.0403, -0.0486,  0.0462,  ...,  0.0861,  0.0025,  0.0432],
        [-0.1275,  0.0479,  0.1841,  ...,  0.0899, -0.1297, -0.0879],
        ...,
        [-0.0445, -0.0548,  0.0123,  ...,  0.1044,  0.0978, -0.0695],
        [ 0.1860,  0.0167,  0.0461,  ..., -0.0963,  0.0785, -0.0225],
        [ 0.0514, -0.0277,  0.0499,  ...,  0.0070,  0.1552,  0.1207]],
       requires_grad=True)

In [7]:
type(gpt2.lm_head.weight)
type(gpt2.transformer.wte.weight)

torch.nn.parameter.Parameter

In [8]:
gpt2.lm_head.weight is gpt2.transformer.wte.weight

True

In [9]:
gpt2.lm_head.weight.data_ptr??

In [10]:
# Returns the address of the first element of :attr:`self` tensor.
gpt2.lm_head.weight.data_ptr() == gpt2.transformer.wte.weight.data_ptr()

True

In [11]:
print(gpt2.state_dict()['transformer.wte.weight'].shape)
print(gpt2.state_dict()['lm_head.weight'].shape)

torch.Size([50257, 768])
torch.Size([50257, 768])


In [13]:
# 只占用一份内存空间
print(gpt2.state_dict()['transformer.wte.weight'])
print(gpt2.state_dict()['lm_head.weight'])

tensor([[-0.1101, -0.0393,  0.0331,  ..., -0.1364,  0.0151,  0.0453],
        [ 0.0403, -0.0486,  0.0462,  ...,  0.0861,  0.0025,  0.0432],
        [-0.1275,  0.0479,  0.1841,  ...,  0.0899, -0.1297, -0.0879],
        ...,
        [-0.0445, -0.0548,  0.0123,  ...,  0.1044,  0.0978, -0.0695],
        [ 0.1860,  0.0167,  0.0461,  ..., -0.0963,  0.0785, -0.0225],
        [ 0.0514, -0.0277,  0.0499,  ...,  0.0070,  0.1552,  0.1207]])
tensor([[-0.1101, -0.0393,  0.0331,  ..., -0.1364,  0.0151,  0.0453],
        [ 0.0403, -0.0486,  0.0462,  ...,  0.0861,  0.0025,  0.0432],
        [-0.1275,  0.0479,  0.1841,  ...,  0.0899, -0.1297, -0.0879],
        ...,
        [-0.0445, -0.0548,  0.0123,  ...,  0.1044,  0.0978, -0.0695],
        [ 0.1860,  0.0167,  0.0461,  ..., -0.0963,  0.0785, -0.0225],
        [ 0.0514, -0.0277,  0.0499,  ...,  0.0070,  0.1552,  0.1207]])


# GPT-2 中的张量绑定 (Tied Tensors) 与权重共享机制

在 GPT-2（以及 LLaMA、Qwen 等现代大语言模型）中，**张量绑定 (Tied Tensors)** 或 **权重共享 (Weight Sharing)** 是一个旨在减少参数量并提升训练效果的核心设计。

以下是该机制的详细拆解：

---

## 1. 核心概念对比：`wte` 与 `lm_head`

在常规的未共享参数的模型中，输入端和输出端是两个完全独立的层。它们在 GPT-2 中的具体职责和结构如下表所示：

| 对比维度 | `transformer.wte` (Word Token Embeddings) | `lm_head` (Language Modeling Head) |
| --- | --- | --- |
| **模型位置** | 模型的 **最前端** (输入层) | 模型的 **最末端** (输出层) |
| **核心作用** | 将用户输入的离散 Token ID（索引）映射为连续的高维向量。 | 将 Transformer 最后一层输出的特征向量映射回词汇表空间，预测下一个词。 |
| **张量形状** | $V \times d_{model}$ (例: `[50257, 768]`) | $V \times d_{model}$ (例: `[50257, 768]`) |
| **数据流向** | Token ID $\rightarrow$ 矩阵查表 $\rightarrow$ $768$维向量 | $768$维向量 $\times$ 权重矩阵 $\rightarrow$ $50257$维 Logits |

*(注：$V$ 为词汇表大小 $50257$，$d_{model}$ 为隐藏层维度 $768$)*

---

## 2. 什么是 Tied Tensors（张量绑定）？

> **定义：** 强行让模型的 `wte` 层（输入词嵌入）和 `lm_head` 层（输出映射头）在物理内存中**共享同一个权重矩阵**。

这就是为什么在您之前提取的代码中，验证这两个层底层内存地址的指针运算会返回 `True`：

```python
# 验证底层内存地址是否完全一致
gpt2.lm_head.weight.data_ptr() == gpt2.transformer.wte.weight.data_ptr()
# 结果: True

```

**运行机制：** 它们在底层只存储了一份数据。当反向传播更新 `wte` 的梯度时，`lm_head` 的权重会自动且同步地更新，反之亦然。

---

## 3. 核心优势：为什么要共享权重？

这种设计并非单纯为了代码整洁，而是有深刻的工程和算法考量，主要包含以下三大优势：

* **大幅减少参数量（节省显存）：**
GPT-2 小型版总参数量约 117M。单单一个词汇表矩阵参数量就是 $50257 \times 768 \approx 38.5M$。若不共享，输入输出层相加会占用约 77M 参数。共享机制直接砍掉了模型近 1/3 的冗余参数，极大地降低了显存压力。
* **语义特征对齐：**
输入层在学习“词在向量空间的位置”，而输出层在学习“根据上下文推断词在向量空间的位置”。两者在语义上互逆且高度一致。共享权重迫使模型学习一套**通用的词汇表征**，通常能有效提升模型的语言理解力。
* **自带正则化（缓解长尾词过拟合）：**
语言模型的词汇表中存在大量低频的长尾词。如果不共享，低频词在 `lm_head` 中的权重可能因极少被触发而无法充分更新。共享机制让低频词无论作为输入还是输出，都能共享梯度的更新，从而使训练更加稳定。

---

## 4. PyTorch 源码实现方式

在 PyTorch 框架中，实现内存级别的权重绑定操作非常直观，直接通过变量赋值即可将指针指向同一块内存区域：

```python
class GPT2LMHeadModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.transformer = GPT2Model(config)
        # 初始化输出头，不使用偏置项 (bias=False)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        
        # 【核心绑定代码】：将 lm_head 的权重指针直接赋予 wte 的权重对象
        self.lm_head.weight = self.transformer.wte.weight

```